In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
# Loading the data 
# Define data paths
data_path = Path('data')

# Load transaction data
train_transaction = pd.read_csv(data_path / 'train_transaction.csv')
print(f"Loaded train_transaction.csv: {train_transaction.shape}")

# Load identity data
train_identity = pd.read_csv(data_path / 'train_identity.csv')
print(f"Loaded train_identity.csv: {train_identity.shape}")


Loaded train_transaction.csv: (590540, 394)
Loaded train_identity.csv: (144233, 41)


In [3]:
# Check how many transactions have identity info
total_transactions = len(train_transaction)
transactions_with_identity = train_transaction['TransactionID'].isin(
    train_identity['TransactionID']).sum()

identity_percentage = (transactions_with_identity / total_transactions) * 100

print(f"\nTransaction Coverage:")
print(f"Total transactions: {total_transactions:,}")
print(f"Transactions with identity: {transactions_with_identity:,}")
print(f"Transactions without identity: {total_transactions - transactions_with_identity:,}")
print(f"Identity coverage: {identity_percentage:.2f}%")


Transaction Coverage:
Total transactions: 590,540
Transactions with identity: 144,233
Transactions without identity: 446,307
Identity coverage: 24.42%


In [4]:
# Check target distribution (isFraud)
fraud_count = train_transaction['isFraud'].sum()
fraud_percentage = (fraud_count / total_transactions) * 100

print(f"\nTarget Distribution (isFraud):")
print(f"Legitimate transactions: {total_transactions - fraud_count:,} ({100-fraud_percentage:.2f}%)")
print(f"Fraudulent transactions: {fraud_count:,} ({fraud_percentage:.2f}%)")
print(f"Class imbalance ratio: 1:{(total_transactions - fraud_count) / fraud_count:.1f}")


Target Distribution (isFraud):
Legitimate transactions: 569,877 (96.50%)
Fraudulent transactions: 20,663 (3.50%)
Class imbalance ratio: 1:27.6


In [5]:
# Create binary flag: 1 if identity info exists, 0 if not
# This captures the PATTERN of missing identity (potential fraud signal!)
train_transaction = train_transaction.assign(
    has_identity=train_transaction['TransactionID'].isin(
        train_identity['TransactionID']
    ).astype(int)
)
print(f"✓ Added 'has_identity' feature")
print(f"  Transactions with identity=1: {train_transaction['has_identity'].sum():,}")
print(f"  Transactions with identity=0: {(train_transaction['has_identity']==0).sum():,}")

✓ Added 'has_identity' feature
  Transactions with identity=1: 144,233
  Transactions with identity=0: 446,307


C:\Users\vedas\AppData\Local\Temp\ipykernel_23312\3344642456.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_transaction = train_transaction.assign(


In [ ]:
# Left join: Keep ALL transactions, add identity info where available
train_data = train_transaction.merge(
    train_identity,
    on='TransactionID',
    how='left',
    validate='one_to_one'  # Ensures no duplicate TransactionIDs
)

print(f"Merge complete!")
print(f"Final dataset shape: {train_data.shape}")
print(f"Total rows: {train_data.shape[0]:,}")
print(f"Total columns: {train_data.shape[1]}")

# Verify no data loss
assert len(train_data) == len(train_transaction), "ERROR: Lost transactions during merge!"
print(f"Verification passed: No transactions lost during merge")

Merge complete!
Final dataset shape: (590540, 435)
Total rows: 590,540
Total columns: 435
Verification passed: No transactions lost during merge


In [8]:
# Calculate missing percentage for each column
missing_stats = pd.DataFrame({
    'column': train_data.columns,
    'missing_count': train_data.isnull().sum(),
    'missing_percentage': (train_data.isnull().sum() / len(train_data)) * 100
})

# Sort by missing percentage
missing_stats = missing_stats.sort_values('missing_percentage', ascending=False)
missing_stats

,column,missing_count,missing_percentage
id_24,id_24,585793,99.196159
id_25,id_25,585408,99.130965
id_08,id_08,585385,99.127070
id_07,id_07,585385,99.127070
id_21,id_21,585381,99.126393
...,...,...,...
C8,C8,0,0.000000
C14,C14,0,0.000000
C13,C13,0,0.000000
C12,C12,0,0.000000


In [9]:
print(f"\nMissing Value Summary:")
print(f"  Columns with 0% missing: {(missing_stats['missing_percentage'] == 0).sum()}")
print(f"  Columns with 1-50% missing: {((missing_stats['missing_percentage'] > 0) & (missing_stats['missing_percentage'] <= 50)).sum()}")
print(f"  Columns with 50-90% missing: {((missing_stats['missing_percentage'] > 50) & (missing_stats['missing_percentage'] <= 90)).sum()}")
print(f"  Columns with >90% missing: {(missing_stats['missing_percentage'] > 90).sum()}")


Missing Value Summary:
  Columns with 0% missing: 21
  Columns with 1-50% missing: 200
  Columns with 50-90% missing: 202
  Columns with >90% missing: 12


In [10]:
# Get columns with >90% missing
high_missing_threshold = 90
columns_to_drop = missing_stats[
    missing_stats['missing_percentage'] > high_missing_threshold
]['column'].tolist()

print(f"\nColumns with >{high_missing_threshold}% missing data: {len(columns_to_drop)}")
if len(columns_to_drop) > 0:
    print("Columns to be dropped:")
    for col in columns_to_drop[:10]:  # Show first 10
        missing_pct = missing_stats[missing_stats['column']==col]['missing_percentage'].values[0]
        print(f"  - {col}: {missing_pct:.2f}% missing")
    if len(columns_to_drop) > 10:
        print(f"  ... and {len(columns_to_drop)-10} more columns")


Columns with >90% missing data: 12
Columns to be dropped:
  - id_24: 99.20% missing
  - id_25: 99.13% missing
  - id_08: 99.13% missing
  - id_07: 99.13% missing
  - id_21: 99.13% missing
  - id_26: 99.13% missing
  - id_27: 99.12% missing
  - id_23: 99.12% missing
  - id_22: 99.12% missing
  - dist2: 93.63% missing
  ... and 2 more columns


In [11]:
train_data.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [13]:
train_data.shape

(590540, 435)